import libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

load dataset

In [3]:
df = pd.read_csv("/content/employee_review_mturk_dataset_test_v6_kaggle.csv")

print("Total rows:", len(df))
print(df.head())

Total rows: 225
      id   person_name                                  nine_box_category  \
0  20051  Lacey Howard  Category 1: 'Risk' (Low performance, Low poten...   
1  20057     Amy Jones  Category 1: 'Risk' (Low performance, Low poten...   
2  20058     Amy Jones  Category 1: 'Risk' (Low performance, Low poten...   
3  20059     Amy Jones  Category 1: 'Risk' (Low performance, Low poten...   
4  20060     Amy Jones  Category 1: 'Risk' (Low performance, Low poten...   

                                            feedback  updated  reviewed  
0  Lacey's performance has been sub standard in t...     True      True  
1  Amy struggles at her work a lot. Shes always o...     True      True  
2  Amy Jones is a nice person and she is dedicate...     True      True  
3  Amy Jones needs to become a better player. She...     True      True  
4  Amy is able to focus on the task at hand only ...     True      True  


clean positive or negative sentiment

In [4]:
def get_sentiment(category_text):
    if "Low performance" in category_text:
        return "Negative"
    else:
        return "Positive"

df["sentiment"] = df["nine_box_category"].apply(get_sentiment)
print("\nSentiment counts:")
print(df["sentiment"].value_counts())


Sentiment counts:
sentiment
Positive    150
Negative     75
Name: count, dtype: int64


clean text feedback

In [5]:
def clean_text(text):
    text = text.lower()
    text = text.replace(".", "")
    text = text.replace(",", "")
    text = text.replace("'", "")
    return text

df["clean_feedback"] = df["feedback"].apply(clean_text)

take input and output

In [6]:
X = df["clean_feedback"]
y = df["sentiment"]   # the category we want to predict

STEP 4: Convert text into numbers (TF-IDF)

In [7]:
vectorizer = TfidfVectorizer()
X_numbers = vectorizer.fit_transform(X)

Split into training and testing data

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_numbers, y, test_size=0.2, random_state=42
)

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


Training rows: 180
Testing rows: 45


train the model on logistic regression

In [9]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

test the model


In [10]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nDetailed Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy: 82.22 %

Detailed Report:
              precision    recall  f1-score   support

    Negative       0.80      0.57      0.67        14
    Positive       0.83      0.94      0.88        31

    accuracy                           0.82        45
   macro avg       0.81      0.75      0.77        45
weighted avg       0.82      0.82      0.81        45



area to improve positive or negative feedback

In [11]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]  # one coefficient per word

# Sort words by their coefficient value
word_scores = pd.DataFrame({
    "word": feature_names,
    "score": coefficients
})

# Most negative words = strongest signal of Negative sentiment
top_negative_words = word_scores.sort_values("score").head(10)
# Most positive words = strongest signal of Positive sentiment
top_positive_words = word_scores.sort_values("score", ascending=False).head(10)

print("\n=== Top words linked to NEGATIVE feedback (areas to improve) ===")
print(top_negative_words["word"].tolist())

print("\n=== Top words linked to POSITIVE feedback (strengths) ===")
print(top_positive_words["word"].tolist())


=== Top words linked to NEGATIVE feedback (areas to improve) ===
['that', 'dylan', 'brayden', 'not', 'show', 'her', 'far', 'so', 'yet', 'valeria']

=== Top words linked to POSITIVE feedback (strengths) ===
['always', 'ella', 'on', 'is', 'hes', 'team', 'our', 'great', 'georgia', 'and']


test on new feedback

In [14]:
new_review = ["He become late to submitt his tasks  and didn't help other"]
new_review_clean = [clean_text(t) for t in new_review]
new_review_numbers = vectorizer.transform(new_review_clean)

prediction = model.predict(new_review_numbers)
print("\nNew review:", new_review[0])
print("Predicted sentiment:", prediction[0])



New review: He become late to submitt his tasks  and didn't help other
Predicted sentiment: Negative
